# Fine-tune ViT-base on FGVC-Aircraft (Google Colab)

**Aviation Intelligence System — ZHAW FS26 Semester Project**

Run this notebook on a free Colab T4 GPU. It will:
1. Install deps
2. Download FGVC-Aircraft (~2 GB)
3. Fine-tune `google/vit-base-patch16-224` for 10 epochs
4. Evaluate top-1 / top-5 / per-manufacturer F1
5. Push the model to Hugging Face Hub for the deployed app to load

**Before running:** in Colab → Runtime → Change runtime type → **T4 GPU**.

You will also need a Hugging Face access token with `write` permission: https://huggingface.co/settings/tokens

In [ ]:
!pip install -q transformers==4.46.0 datasets==3.0.0 accelerate==1.0.0 torchvision==0.20.1 evaluate==0.4.3 huggingface_hub

In [ ]:
import torch, os
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste your HF write token

## 1. Download FGVC-Aircraft

In [ ]:
from torchvision.datasets import FGVCAircraft
from pathlib import Path
DATA = Path('/content/data')
DATA.mkdir(exist_ok=True)
for split in ('train', 'val', 'test'):
    FGVCAircraft(root=str(DATA), split=split, annotation_level='variant', download=True)
print('Done.')

## 2. Dataset wrapper

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import ViTImageProcessor

MODEL_ID = 'google/vit-base-patch16-224'
processor = ViTImageProcessor.from_pretrained(MODEL_ID)

class FGVCWrapper(Dataset):
    def __init__(self, split, train):
        self.ds = FGVCAircraft(root=str(DATA), split=split, annotation_level='variant', download=False)
        if train:
            self.tx = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomResizedCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.2, 0.2, 0.2),
            ])
        else:
            self.tx = transforms.Compose([transforms.Resize((224, 224))])
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        img, lbl = self.ds[i]
        img = self.tx(img.convert('RGB'))
        pix = processor(img, return_tensors='pt')['pixel_values'][0]
        return {'pixel_values': pix, 'labels': int(lbl)}

train_ds = FGVCWrapper('train', True)
val_ds   = FGVCWrapper('val', False)
test_ds  = FGVCWrapper('test', False)
classes  = train_ds.ds.classes
print(f'classes: {len(classes)}, train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')

## 3. Train

In [ ]:
import numpy as np
from transformers import ViTForImageClassification, Trainer, TrainingArguments

model = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(classes),
    id2label={i: c for i, c in enumerate(classes)},
    label2id={c: i for i, c in enumerate(classes)},
    ignore_mismatched_sizes=True,
)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, -1)
    top5 = np.argsort(logits, -1)[:, -5:]
    return {
        'accuracy': float((preds == labels).mean()),
        'top5_accuracy': float(np.mean([l in t for l, t in zip(labels, top5)])),
    }

args = TrainingArguments(
    output_dir='/content/aircraft-vit',
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=True,
    report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=metrics)
trainer.train()

## 4. Evaluate on test split

In [ ]:
test_metrics = trainer.evaluate(test_ds)
print(test_metrics)

In [ ]:
# per-manufacturer F1 + confusion
import torch
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns, numpy as np

preds_all, labels_all = [], []
model.eval().cuda()
from torch.utils.data import DataLoader
dl = DataLoader(test_ds, batch_size=64, num_workers=2)
with torch.no_grad():
    for batch in dl:
        out = model(pixel_values=batch['pixel_values'].cuda()).logits.argmax(-1).cpu().numpy()
        preds_all.extend(out); labels_all.extend(batch['labels'].numpy())

print(classification_report(labels_all, preds_all, target_names=classes, zero_division=0))
cm = confusion_matrix(labels_all, preds_all)
plt.figure(figsize=(14, 12)); sns.heatmap(cm, cmap='magma'); plt.title('Confusion matrix'); plt.show()

## 5. Push to Hugging Face Hub

In [ ]:
REPO = 'dubattim/aviation-intelligence-vit-fgvc'  # change if you want a different repo
trainer.save_model('/content/aircraft-vit')
processor.save_pretrained('/content/aircraft-vit')
model.push_to_hub(REPO)
processor.push_to_hub(REPO)
print(f'Done → https://huggingface.co/{REPO}')

After this finishes, update `src/cv/infer.py` `HF_FALLBACK` constant in the repo to point at your new model id, then redeploy the Space.